# 04 — Model A pilot: G0 vs A-GRU (Day 3 Step 1)

Runs exactly the same code as `uv run python -m bdsl.cli train-text2gloss` (see `CLAUDE.md` → COMMANDS).

- **Guard:** training refuses to start unless `gloss-validate` is clean (ADR-0024). Mock rows and rows whose notes say "not video-verified" are refused by default.
- **`--allow-test-data` is for a practice run only.** It accepts fake review data; every output is then named `TEST_…` and says `TEST_RUN: true`. Never report such numbers as Model A results.
- **Pilot numbers are a "pipeline test"** (5 val / 5 test sentences): they show the code runs and the model can overfit, not that it generalises (ADR-0020).
- Outputs go to `work/runs/` (gitignored). Checkpoints resume with `--resume <run folder>`.


In [ ]:
from bdsl.cli import main
from bdsl.config import load_config

cfg = load_config(data="pilot50")
print("env:", cfg["_meta"]["env"], "| runs:", cfg["paths"]["runs_dir"])
print(cfg["text2gloss"])

In [ ]:
# The check the training guard uses. Real runs: no option, and it must report no problems.
print("exit code:", main(["gloss-validate"]))

In [ ]:
# Overfit check on 1 training sentence (add "--allow-test-data" ONLY for a practice run on fake data).
print("exit code:", main(["train-text2gloss", "--overfit", "1"]))

In [ ]:
# Full pilot run: G0 vs A-GRU, 3 seeds, early stopping on val gloss WER, beam 4 at the end.
print("exit code:", main(["train-text2gloss"]))

In [ ]:
# The newest run's metrics and per-item table.
import json
from pathlib import Path

import pandas as pd

run = max(Path(cfg["paths"]["runs_dir"]).glob("*text2gloss_pilot50_*"), key=lambda p: p.stat().st_mtime)
prefix = "TEST_" if run.name.startswith("TEST_") else ""
print(run.name)
print(json.dumps(json.loads((run / f"{prefix}metrics.json").read_text(encoding="utf-8"))["agru_mean_sd"], indent=1))
pd.read_csv(run / f"{prefix}per_item.csv", encoding="utf-8-sig")